# A2 — Recommender Systems với Spark Structured Streaming

Notebook trình bày hai nội dung lý thuyết của A2 theo hướng từ nền tảng đến chuyên sâu, sau đó dùng một **synthetic clickstream có chủ đích** để minh họa. Các thuật ngữ chuyên ngành như *recommender system*, *streaming*, *event time*, *watermark*, *window*, *state* và *checkpoint* được giữ bằng tiếng Anh.

> **Phạm vi:** A2 nói về recommender system nói chung. Retail Rocket, Kafka cluster và full real-data pipeline thuộc Part B; cuối notebook có bảng ánh xạ sang Part B.

## Mục lục

- [0. Giới thiệu và phạm vi](#sec-0)
- [1. Hệ thống gợi ý thời gian thực](#sec-1)
  - [1.1 Recommender system là gì?](#sec-1-1)
  - [1.2 Các thành phần của recommender system](#sec-1-2)
  - [1.3 Real-time recommender là gì?](#sec-1-3)
  - [1.4 Vì sao continuous event stream cải thiện personalization?](#sec-1-4)
  - [1.5 Batch và streaming recommendation](#sec-1-5)
  - [1.6 Các cấp độ cập nhật personalization](#sec-1-6)
  - [1.7 Các họ thuật toán recommendation](#sec-1-7)
  - [1.8 Explicit và implicit feedback](#sec-1-8)
  - [1.9 Các kiến trúc batch/streaming kết hợp](#sec-1-9)
  - [1.10 Hạn chế và phương pháp đánh giá](#sec-1-10)
  - [1.11 Minh họa batch và real-time context](#sec-1-11)
- [2. Xử lý clickstream bằng PySpark Structured Streaming](#sec-2)
  - [2.1 Clickstream và các loại hành vi](#sec-2-1)
  - [2.2 Continuous ingestion](#sec-2-2)
  - [2.3 Event time và processing time](#sec-2-3)
  - [2.4 Theo dõi hành vi bằng window](#sec-2-4)
  - [2.5 Watermark và late data](#sec-2-5)
  - [2.6 Biểu diễn hành vi thành intent score](#sec-2-6)
  - [2.7 Phát hiện high-intent user](#sec-2-7)
  - [2.8 Kích hoạt immediate recommendation](#sec-2-8)
  - [2.9 Output mode và foreachBatch](#sec-2-9)
  - [2.10 Stateful processing](#sec-2-10)
  - [2.11 Fault tolerance](#sec-2-11)
  - [2.12 Serving, evaluation và thách thức](#sec-2-12)
- [3. Thực hành PySpark](#sec-3)
- [4. Kết luận](#sec-4)
- [5. Tài liệu tham khảo](#sec-5)
- [Phụ lục — Liên hệ với Part B](#appendix)

<a id="sec-0"></a>
## 0. Giới thiệu và phạm vi

Recommender system giúp người dùng tìm item phù hợp trong một không gian lựa chọn lớn: sản phẩm, phim, bài hát, tin tức, khóa học hoặc quảng cáo. Một hệ thống truyền thống thường học từ lịch sử theo batch. Tuy nhiên, preference và context có thể thay đổi ngay trong một session; hành vi vừa xảy ra có thể quan trọng hơn lịch sử nhiều ngày trước.

A2 trả lời hai câu hỏi:

1. **Real-time recommender là gì, và streaming phản ứng với hành vi mới tốt hơn batch như thế nào?**
2. **PySpark Structured Streaming có thể nhận clickstream, duy trì session context, tính live intent và kích hoạt recommendation như thế nào?**

Phần thực hành chỉ nhằm làm rõ lý thuyết. Nó dùng file replay và synthetic data nhỏ để không phụ thuộc Kafka, Docker hoặc dataset thật.

<a id="sec-1"></a>
# 1. Hệ thống gợi ý thời gian thực

<a id="sec-1-1"></a>
## 1.1 Recommender system là gì?

Một **recommender system** ước lượng mức độ phù hợp của item (i) với user (u) trong context (c):

$$
\hat r(u,i,c)
$$

Sau đó hệ thống trả về top-(K) item có score cao nhất. Recommendation không chỉ là prediction: một hệ thống hoàn chỉnh còn phải tạo candidate, xếp hạng, áp dụng business rules, phục vụ kết quả và thu nhận feedback.

<a id="sec-1-2"></a>
## 1.2 Các thành phần của recommender system

```text
[User history] ─┐
                ├──> [Context / features] ──> [Candidate generation]
[Current events]┘                                   │
[Recommendation model] ─────────────────────────────┘
                                                     ↓
                              [Scoring] → [Ranking + rules] → [Top-K]
                                                           ↓
                                                     [User feedback]
```

- **Candidate generation:** thu hẹp hàng triệu item thành một tập nhỏ.
- **Scoring:** ước lượng relevance của từng candidate.
- **Ranking:** sắp xếp và áp dụng diversity, availability hoặc policy.
- **Serving:** đưa kết quả tới website/app với latency thấp.
- **Feedback loop:** hành vi sau recommendation trở thành dữ liệu mới.

<a id="sec-1-3"></a>
## 1.3 Real-time recommender là gì?

Một **real-time recommender** liên tục nhận hành vi mới, cập nhật context hiện tại và tạo recommendation khi user vẫn đang hoạt động. Cần phân biệt:

| Khái niệm | Nội dung |
|---|---|
| Model training | Học parameters từ historical data; thường là batch |
| Context/feature update | Cập nhật item vừa xem, cart, session intent; phù hợp streaming |
| Online inference/scoring | Dùng model và live context để xếp hạng candidate |
| Online learning | Cập nhật model parameters liên tục; phức tạp hơn và không bắt buộc |

Thiết kế phổ biến là **offline model + online scoring**: phần nặng được train định kỳ, còn stream làm personalization phản ứng với context mới.

<a id="sec-1-4"></a>
## 1.4 Vì sao continuous event stream cải thiện personalization?

Purchase intent và content interest có “half-life”: user có thể chuyển mục tiêu trong vài phút. Stream giúp hệ thống:

- thấy current-session behaviour thay vì chờ lần batch tiếp theo;
- phục vụ user mới bằng session context dù chưa có long-term profile;
- cập nhật feature và aggregate gia tăng thay vì tính lại toàn bộ;
- phản hồi trước khi user rời ứng dụng.


```text
10:00  User ──view A────> Application ──event──> Streaming recommender
10:02  User ──addcart A─> Application ──event──> high intent → top-K → User
10:08  User rời ứng dụng
02:00 hôm sau: nightly batch mới xử lý các event trên
```

Streaming không tự động làm model “thông minh hơn”; lợi ích trực tiếp là **freshness** và khả năng phản ứng.

<a id="sec-1-5"></a>
## 1.5 Batch và streaming recommendation

| Tiêu chí | Batch recommendation | Streaming recommendation |
|---|---|---|
| Data | Bounded snapshot | Unbounded event stream |
| Freshness | Giờ hoặc ngày | Giây hoặc một micro-batch |
| Current session | Chưa thấy cho tới lần chạy sau | Cập nhật liên tục |
| User mới | Thiếu history | Có thể dùng session context |
| Computation | Model nặng, retraining | Feature update và scoring nhẹ |
| State | Tính lại hoặc snapshot | Incremental state |
| Late/out-of-order data | Ít quan trọng hơn | Cần event time và watermark |
| Recovery | Chạy lại job | Replayable source + checkpoint |
| Cost | Bursty | Always-on |

Hai cách xử lý bổ sung cho nhau: batch tối ưu model từ lịch sử; streaming làm model đó phản ứng với hành vi vừa xuất hiện.

<a id="sec-1-6"></a>
## 1.6 Các cấp độ cập nhật personalization

| Cấp độ | Cập nhật liên tục | Ví dụ |
|---|---|---|
| 1. Live context/features | Session items, intent, last category | Windowed intent |
| 2. Model statistics | Popularity, counts, co-occurrence | Trending trong 10 phút |
| 3. Model parameters | Embedding hoặc online weights | Online learning/retraining |

DataFrame-based Spark ML như ALS huấn luyện trên bounded DataFrame. Vì vậy kiến trúc thực tế thường dùng Structured Streaming cho cấp 1–2 và batch retraining cho cấp 3. Các estimator online cũ như `StreamingKMeans` thuộc legacy DStream/RDD API.

<a id="sec-1-7"></a>
## 1.7 Các họ thuật toán recommendation

| Phương pháp | Ý tưởng | Điểm mạnh | Hạn chế | Fit với streaming |
|---|---|---|---|---|
| Popularity | Item phổ biến nhất | Đơn giản, fallback tốt | Không cá nhân hóa | Rất tốt |
| Content-based | Tương tự theo thuộc tính | Hỗ trợ item mới | Cần metadata | Tốt |
| User-based CF | User giống nhau | Khai thác taste | Sparse, user thay đổi nhanh | Khó |
| Item-to-item CF | Item thường được dùng cùng | Ổn định, explainable | Item cold start | Rất tốt |
| ALS / matrix factorization | Latent user-item vectors | Mạnh với history đủ dày | Batch retraining | Trung bình |
| Session-based | Dự đoán item kế tiếp | Hợp user ẩn danh | Serving phức tạp | Tốt |
| Hybrid | Kết hợp nhiều signal | Robust | Nhiều moving parts | Phổ biến |

Demo sử dụng **item-to-item CF** vì similarity có thể precompute, sau đó lookup nhanh từ seed item của session hiện tại.

<a id="sec-1-8"></a>
## 1.8 Explicit và implicit feedback

- **Explicit feedback:** rating, like/dislike hoặc review.
- **Implicit feedback:** view, click, add-to-cart, play, purchase.

Implicit event biểu diễn **confidence**, không phải rating. Add-to-cart thường mạnh hơn view, nhưng không chứng minh user thích item; transaction cũng có thể bị ảnh hưởng bởi giá, tồn kho hoặc khuyến mãi. Không có event không phải là negative label.

Thiết kế intent vì vậy dùng weighted evidence trong một khoảng thời gian gần, thay vì coi mọi item chưa tương tác là “không thích”.

<a id="sec-1-9"></a>
## 1.9 Các kiến trúc batch/streaming kết hợp

| Kiến trúc | Mô tả | Trade-off |
|---|---|---|
| Batch-only | Train và xuất recommendation định kỳ | Đơn giản nhưng stale |
| Lambda | Batch layer + speed layer | Fresh và accurate nhưng hai luồng logic |
| Kappa | Mọi thứ là stream, replay log khi cần | Một code path nhưng replay dài tốn kém |
| Offline model + online scoring | Batch train model, stream tính context và score | Cân bằng freshness/complexity |

```text
[Historical data] → [Offline model training] → [Item similarity] ─┐
                                                                    ├→ [Online scoring] → [Top-K] → [Serving]
[Live events]     → [Streaming context + intent] ──────────────────┘
[Live events]     → [Recent popularity] ───────────────────────────┘
```

Notebook chọn phương án cuối vì cùng một Spark ecosystem có thể xử lý batch model và streaming context.

<a id="sec-1-10"></a>
## 1.10 Hạn chế và phương pháp đánh giá

**Hạn chế:** user/item cold start, popularity bias, filter bubble, feedback loop, stale model, bot/skewed user và availability constraints.

**Offline metrics:** Precision@K, Recall@K, HitRate@K, NDCG và coverage. Offline replay chỉ là proxy, không chứng minh causal uplift.

**Online metrics:** CTR, add-to-cart rate, conversion, revenue/session và retention qua A/B test.

**System metrics:** end-to-end latency, micro-batch duration, consumer lag, state size và error rate.

<a id="sec-1-11"></a>
## 1.11 Minh họa batch và real-time context

Ví dụ dưới đây không dùng Spark. Mục tiêu là nhìn thấy khác biệt: batch giữ nguyên score cho tới lần refresh, trong khi streaming context cập nhật sau từng event.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

context_demo = pd.DataFrame([
    ("10:00", "view", 1),
    ("10:01", "view", 1),
    ("10:02", "addtocart", 5),
], columns=["time", "event", "weight"])
context_demo["streaming_intent"] = context_demo["weight"].cumsum()
context_demo["batch_intent"] = 0  # snapshot gần nhất chưa chứa các event mới

fig, ax = plt.subplots(figsize=(8, 3))
ax.step(context_demo["time"], context_demo["streaming_intent"], where="post",
        marker="o", label="Streaming context", color="#2878b5")
ax.step(context_demo["time"], context_demo["batch_intent"], where="post",
        marker="o", label="Batch snapshot", color="#999999")
ax.axhline(5, linestyle="--", color="#d9534f", label="High-intent threshold")
ax.set(title="Streaming phản ứng ngay khi context thay đổi",
       xlabel="Event time", ylabel="Intent score")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()
context_demo

<a id="sec-2"></a>
# 2. Xử lý clickstream bằng PySpark Structured Streaming

<a id="sec-2-1"></a>
## 2.1 Clickstream và các loại hành vi

**Clickstream** là chuỗi event mô tả user tương tác với ứng dụng. Một schema tổng quát:

```json
{
  "timestamp": 1710000000000,
  "userid": 1024,
  "event": "addtocart",
  "itemid": 355908,
  "transactionid": null
}
```

Demo sử dụng ba event:

| Event | Evidence | Weight |
|---|---|---:|
| `view` | Quan tâm nhẹ | 1 |
| `addtocart` | Ý định mạnh | 5 |
| `transaction` | Conversion đã xảy ra | 10 |

`transactionid` chỉ tồn tại khi event là transaction.

<a id="sec-2-2"></a>
## 2.2 Continuous ingestion

Trong production, producer ghi event vào Kafka; Spark dùng `readStream`, parse JSON bằng explicit schema và tạo streaming DataFrame.

```text
[Web / mobile app] → [Event producer] → [Kafka partitions]
        → [Spark readStream] → [from_json + schema]
        → [Event-time transformations] → [Stateful aggregation]
        → [foreachBatch] → [Recommendation sink]
```

Kafka key thường là user ID để event của cùng user đi vào cùng partition. File replay có thể tạo cùng cột `value`, nên mọi transformation phía sau source được tái sử dụng.

<a id="sec-2-3"></a>
## 2.3 Event time và processing time

| | Event time | Processing time |
|---|---|---|
| Ý nghĩa | Khi hành vi xảy ra | Khi Spark xử lý |
| Bị ảnh hưởng bởi network lag | Không | Có |
| Deterministic khi replay | Có | Không |
| Dùng cho | Business window, cooldown | Trigger và monitoring |

Nếu add-to-cart xảy ra 10:02 nhưng đến Spark lúc 10:05, nó vẫn thuộc business window của 10:02. Vì vậy intent phải dựa trên event time, không dựa trên `current_timestamp()`.

<a id="sec-2-4"></a>
## 2.4 Theo dõi hành vi bằng window

| Window | Đặc điểm | Use case |
|---|---|---|
| Tumbling | Không overlap | Báo cáo theo kỳ |
| Sliding | Overlap; đánh giá lại định kỳ | Rolling intent |
| Session | Đóng sau inactivity gap | Session analytics |

A2 dùng window dài **10 phút**, slide **5 phút**. Mỗi 5 phút hệ thống đánh giá lại 10 phút gần nhất; một event thường thuộc hai window.

```text
event at 10:07       ●
10:00–10:10    ├──────────┤
10:05–10:15         ├──────────┤
```

State được quản lý theo key và window, không cần giữ toàn bộ lịch sử user.

<a id="sec-2-5"></a>
## 2.5 Watermark và late data

`withWatermark("event_time", "5 minutes")` cho Spark biết mức độ late data dự kiến:

$$
\text{watermark} \approx \max(\text{event time seen}) - 5\text{ minutes}
$$

Khi watermark vượt qua end của window, Spark có thể finalise và xóa state. Event quá trễ cho window đã đóng bị bỏ.

| Watermark | Late-event tolerance | State | Finalisation latency |
|---|---|---|---|
| Ngắn | Thấp | Nhỏ | Thấp |
| Dài | Cao | Lớn | Cao |

Watermark không phải phép lọc đơn giản trên từng event; quyết định còn phụ thuộc window end và stateful operator.

<a id="sec-2-6"></a>
## 2.6 Biểu diễn hành vi thành intent score

Với user (u), item (i), window (W):

$$
\mathrm{Intent}(u,i,W)=\sum_{e\in E(u,i),\,t_e\in W}w(e)
$$

và score ở user level:

$$
\mathrm{UserIntent}(u,W)=\sum_i \mathrm{Intent}(u,i,W)
$$

Item-level score dùng để chọn seed và trigger. User-level score chỉ là context/report, không thay thế item-level rule.

**Các lựa chọn khác:** exponential time decay, session score hoặc learned propensity model. Weighted sliding-window sum được chọn vì explainable, đúng yêu cầu và được Spark quản lý state trực tiếp.

<a id="sec-2-7"></a>
## 2.7 Phát hiện high-intent user

Rule minh họa:

$$
\mathrm{Intent}(u,i,W)\geq 5 \quad\land\quad \mathrm{Transactions}(u,i,W)=0
$$

| Hành vi trong window | Score | Trigger |
|---|---:|---|
| 2 views | 2 | Không |
| 5 views | 5 | Có — heavy browsing |
| 1 add-to-cart | 5 | Có — cart without purchase observed yet |
| add-to-cart + transaction | 15 | Không — đã conversion |

`cart_no_purchase` không có nghĩa là confirmed cart abandonment; nó chỉ nói chưa thấy transaction trong window hiện tại. Threshold production cần calibration và A/B test.

<a id="sec-2-8"></a>
## 2.8 Kích hoạt immediate recommendation

Item đạt rule trở thành **seed item**. Candidate score:

$$
\mathrm{RecScore}(u,j,W)=\sum_{i\in \mathrm{Seeds}(u,W)}
\mathrm{Similarity}(i,j)\times\mathrm{Intent}(u,i,W)
$$

```text
[Windowed item intent]
          ↓  score ≥ 5 and transactions = 0
      [Seed item] → [Item similarity] → [Exclude seen / policy] → [Top-K]
                         │
                         └── no neighbour ──> [Popularity fallback] ─┘
```

Production còn cần seen-item exclusion, seed/recommendation cooldown, stock/policy filter và idempotent delivery.

<a id="sec-2-9"></a>
## 2.9 Output mode và foreachBatch

| Mode | Output | Phù hợp |
|---|---|---|
| Append | Window đã finalise | Curated history |
| Update | Aggregate vừa thay đổi | Low-latency trigger |
| Complete | Toàn bộ result table | Dashboard nhỏ |

Demo dùng **update mode** để không chờ watermark đóng window.

`foreachBatch` biến mỗi micro-batch output thành static DataFrame, cho phép:

- non-time window function như `row_number()` để lấy top-K;
- stream-static join với item similarity;
- fallback, logging và nhiều sink;
- dùng `batch_id` làm idempotency key.

<a id="sec-2-10"></a>
## 2.10 Stateful processing

| API | State | Use case |
|---|---|---|
| Windowed aggregation | Spark-managed, watermark eviction | Intent trong demo |
| applyInPandasWithState | Arbitrary per-key state | Custom session/decay |
| transformWithStateInPandas | Multiple state variables, TTL/timers | State machine phức tạp |
| mapGroupsWithState | JVM API | Custom state trên Scala/Java |

Windowed aggregation là lựa chọn nhỏ nhất đáp ứng A2. Custom state chỉ cần khi business logic không biểu diễn được bằng window aggregate.

<a id="sec-2-11"></a>
## 2.11 Fault tolerance

- **Replayable source:** file list hoặc Kafka offsets cho phép đọc lại input chưa commit.
- **Checkpoint:** lưu offsets, commits và state.
- **State store:** giữ aggregate theo key/window; watermark giới hạn kích thước.
- **Sink idempotency:** `foreachBatch` là at-least-once, nên output phải deduplicate theo `batch_id` hoặc business key.

Ghi Parquet và sent-log bằng hai write độc lập chỉ cho **idempotent/effectively-once output**, không phải end-to-end exactly-once transaction.

<a id="sec-2-12"></a>
## 2.12 Serving, evaluation và thách thức

| Target | Vai trò |
|---|---|
| Kafka recommendation topic | Decouple website/push consumer |
| Redis/Cassandra | Low-latency lookup theo user |
| Parquet/Delta | Analytics, audit, retraining |
| Console | Debug/classroom demo |

Cần theo dõi input rows, trigger duration, watermark, state rows, dropped-by-watermark, triggered users và recommendation count. Các thách thức gồm late data, duplicate trigger, state growth, hot key/bot, stale model, cold start và feedback loop.

<a id="sec-3"></a>
# 3. Thực hành PySpark

Mục tiêu của demo:

1. Tạo historical interactions và train một item-to-item CF nhỏ.
2. Tạo các trường hợp clickstream xác định trước.
3. Quan sát event-time window và live intent.
4. Phát hiện high intent và tạo top-3 recommendation trong `foreachBatch`.
5. Bơm accepted-late và too-late event.
6. Restart bằng cùng checkpoint và chứng minh không đọc lại file cũ.

> Demo dùng file source. Nếu native Windows gặp `NativeIO$Windows.access0`, chạy notebook trong Spark container của repo; đây là giới hạn môi trường, không phải thay đổi logic.

In [ ]:
# 3.1 Cấu hình demo
import os, sys, json, time, shutil, logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, StringType

DEMO_ROOT = Path("work/a2_v3_demo")
INPUT_DIR = DEMO_ROOT / "stream_in"
CHECKPOINT_DIR = DEMO_ROOT / "checkpoint"
OUTPUT_DIR = DEMO_ROOT / "recommendations"
LOG_DIR = DEMO_ROOT / "logs"
RESET_DEMO = os.environ.get("RESET_A2_DEMO", "true").lower() == "true"
PYTHON_EXE = os.environ.get("PYSPARK_PYTHON", getattr(sys, "_base_executable", sys.executable))
os.environ["PYSPARK_PYTHON"] = PYTHON_EXE

if RESET_DEMO:
    shutil.rmtree(DEMO_ROOT, ignore_errors=True)
for path in (INPUT_DIR, CHECKPOINT_DIR, OUTPUT_DIR, LOG_DIR):
    path.mkdir(parents=True, exist_ok=True)

spark = (SparkSession.builder
    .appName("A2-Recommender-Theory-Demo")
    .master(os.environ.get("SPARK_MASTER", "local[2]"))
    .config("spark.driver.memory", os.environ.get("DRIVER_MEMORY", "4g"))
    .config("spark.pyspark.python", PYTHON_EXE)
    .config("spark.executorEnv.PYSPARK_PYTHON", PYTHON_EXE)
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| demo root:", DEMO_ROOT.resolve())

## 3.2 Synthetic data có chủ đích

Demo dùng event xác định trước, không dùng Retail Rocket. Mục đích là để từng kết quả có thể giải thích được và assertion có thể lặp lại.

In [ ]:
# 3.2 Synthetic data có chủ đích
WEIGHTS = {"view": 1, "addtocart": 5, "transaction": 10}
WINDOW, SLIDE, WATERMARK = "10 minutes", "5 minutes", "5 minutes"
INTENT_THRESHOLD, TOP_K = 5, 3
BASE = pd.Timestamp("2024-01-01T10:00:00Z")



### 3.2.1 Historical interactions cho offline model

Ba cụm item đồng xuất hiện tạo dữ liệu nhỏ cho item-to-item collaborative filtering. Đây là historical/batch data, khác với live clickstream.

In [ ]:
# Historical baskets: ba nhóm item thường xuất hiện cùng nhau.
basket_patterns = [
    [101, 102, 103], [101, 102], [101, 103],
    [201, 202, 203], [201, 202], [201, 203],
    [301, 302, 303], [301, 302], [301, 303],
]
history_rows = []
for userid in range(1, 73):
    basket = basket_patterns[(userid - 1) % len(basket_patterns)]
    for pos, itemid in enumerate(basket):
        history_rows.append((userid, itemid, "view"))
        if (userid + pos) % 4 == 0:
            history_rows.append((userid, itemid, "addtocart"))
HISTORY_SCHEMA = StructType([
    StructField("userid", IntegerType(), False),
    StructField("itemid", IntegerType(), False),
    StructField("event", StringType(), False),
])
history_path = DEMO_ROOT / "history.json"
history_path.write_text("\n".join(
    json.dumps({"userid": u, "itemid": i, "event": e})
    for u, i, e in history_rows
) + "\n", encoding="utf-8")
history_df = spark.read.schema(HISTORY_SCHEMA).json(str(history_path))



### 3.2.2 Event factory và tập case cơ sở

`event_at` biến số phút thành epoch milliseconds. Bốn nhóm đầu dùng cho static preview/fallback; case watermark và restart được replay riêng.

In [ ]:
def event_at(minutes, userid, event, itemid, transactionid=None):
    timestamp = int((BASE + pd.Timedelta(minutes=minutes)).timestamp() * 1000)
    return (timestamp, userid, event, itemid, transactionid)

# Batch đầu có đủ bốn case để validation không phụ thuộc random data.
initial_events = [
    event_at(0, 9001, "view", 101),
    event_at(1, 9001, "view", 101),
    event_at(2, 9001, "addtocart", 101),       # score 7 -> trigger
    event_at(0, 9002, "view", 201),
    event_at(.5, 9002, "view", 201),
    event_at(1, 9002, "view", 201),
    event_at(1.5, 9002, "view", 201),
    event_at(2, 9002, "view", 201),            # score 5 -> trigger
    event_at(1, 9003, "addtocart", 301),
    event_at(2, 9003, "transaction", 301, 7001), # conversion -> suppress
    event_at(1, 9004, "view", 102),
    event_at(2, 9004, "view", 102),            # score 2 -> no trigger
]
advance_events = [event_at(20, 9999, "view", 101)]
late_events = [
    event_at(18, 9010, "addtocart", 103),      # trong watermark -> accepted
    event_at(0, 9011, "addtocart", 203),       # quá trễ sau khi max=20 -> dropped
]
restart_events = [event_at(22, 9020, "addtocart", 202)]

event_columns = ["timestamp", "userid", "event", "itemid", "transactionid"]
initial_pdf = pd.DataFrame(initial_events, columns=event_columns)
display(initial_pdf.assign(event_time=pd.to_datetime(initial_pdf.timestamp, unit="ms", utc=True)))

In [ ]:
# 3.3 Visualization: clickstream theo user
timeline = initial_pdf.copy()
timeline["event_time"] = pd.to_datetime(timeline["timestamp"], unit="ms", utc=True)
symbols = {"view": "o", "addtocart": "D", "transaction": "s"}
colors = {"view": "#2878b5", "addtocart": "#f28e2b", "transaction": "#2ca02c"}

fig, ax = plt.subplots(figsize=(10, 3.6))
for event_name in symbols:
    part = timeline[timeline.event == event_name]
    ax.scatter(part.event_time, part.userid, marker=symbols[event_name],
               color=colors[event_name], s=65, label=event_name)
ax.set(title="Synthetic clickstream: các case được thiết kế trước",
       xlabel="Event time", ylabel="User ID")
ax.legend(ncol=3); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

## 3.4 Offline item-to-item CF

Tái sử dụng ý tưởng từ hai notebook cũ: với mỗi user-item lấy interaction mạnh nhất; co-occurrence được chuẩn hóa bằng weighted popularity trên cùng active-user population.

[
sim(i,j)=
rac{sum_u min(r_{u,i},r_{u,j})}{sqrt{pop(i)pop(j)}}
]

Model chỉ nhìn historical data, không nhìn live events.

In [ ]:
# 3.4 Train item similarity và popularity fallback
weight_expr = (F.when(F.col("event") == "view", 1)
               .when(F.col("event") == "addtocart", 5)
               .when(F.col("event") == "transaction", 10)
               .otherwise(0))

user_item = (history_df.withColumn("w", weight_expr)
             .groupBy("userid", "itemid").agg(F.max("w").alias("w")))
active_users = (user_item.groupBy("userid").count()
                .filter("count BETWEEN 2 AND 100").select("userid"))
ui = user_item.join(active_users, "userid")

a, b = ui.alias("a"), ui.alias("b")
pairs = (a.join(b, "userid")
         .filter(F.col("a.itemid") != F.col("b.itemid"))
         .groupBy(F.col("a.itemid").alias("item_a"),
                  F.col("b.itemid").alias("item_b"))
         .agg(F.sum(F.least("a.w", "b.w")).alias("co"),
              F.count("*").alias("co_users")))
pop = ui.groupBy("itemid").agg(F.sum("w").alias("pop"))
item_sim = (pairs
    .join(pop.select(F.col("itemid").alias("item_a"), F.col("pop").alias("pop_a")), "item_a")
    .join(pop.select(F.col("itemid").alias("item_b"), F.col("pop").alias("pop_b")), "item_b")
    .withColumn("similarity", F.col("co") / F.sqrt(F.col("pop_a") * F.col("pop_b")))
    .select("item_a", "item_b", "similarity", "co_users").cache())
popular = (pop.orderBy(F.desc("pop"), "itemid").limit(6)
           .select(F.col("itemid").alias("rec_item"),
                   F.col("pop").alias("pop_score")).cache())

max_similarity = item_sim.agg(F.max("similarity")).first()[0]
assert max_similarity <= 1.000001
print("item_sim rows:", item_sim.count(), "| max similarity:", round(max_similarity, 4))
item_sim.orderBy("item_a", F.desc("similarity")).show(20, truncate=False)

# Visualization: similarity matrix
sim_pdf = item_sim.toPandas()
matrix = sim_pdf.pivot(index="item_a", columns="item_b", values="similarity").fillna(0)
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=45)
ax.set_yticks(range(len(matrix.index)), matrix.index)
ax.set(title="Item-to-item similarity", xlabel="item_b", ylabel="item_a")
fig.colorbar(image, ax=ax, label="similarity")
plt.tight_layout(); plt.show()

In [ ]:
# 3.5 Static preview: event thuộc window nào và intent thay đổi ra sao
EVENT_SCHEMA = StructType([
    StructField("timestamp", LongType(), False),
    StructField("userid", IntegerType(), False),
    StructField("event", StringType(), False),
    StructField("itemid", IntegerType(), False),
    StructField("transactionid", IntegerType(), True),
])
static_path = DEMO_ROOT / "initial_static.json"
initial_pdf.to_json(static_path, orient="records", lines=True)
static_events = spark.read.schema(EVENT_SCHEMA).json(str(static_path))
static_intent = (static_events
    .withColumn("event_time", F.timestamp_millis("timestamp"))
    .withColumn("weight", weight_expr)
    .groupBy(F.window("event_time", WINDOW, SLIDE).alias("window"), "userid", "itemid")
    .agg(F.sum("weight").alias("intent_score"),
         F.sum((F.col("event") == "view").cast("int")).alias("views"),
         F.sum((F.col("event") == "addtocart").cast("int")).alias("carts"),
         F.sum((F.col("event") == "transaction").cast("int")).alias("transactions"))
    .orderBy("userid", "window.start"))

static_intent.show(30, truncate=False)

intent_pdf = static_intent.filter("userid IN (9001, 9002, 9003, 9004)").toPandas()
intent_pdf["label"] = (intent_pdf["userid"].astype(str) + "\n" +
                       intent_pdf["window"].map(lambda w: w["start"].strftime("%H:%M")))
fig, ax = plt.subplots(figsize=(10, 3.5))
bar_colors = ["#d9534f" if s >= INTENT_THRESHOLD and t == 0 else "#6baed6"
              for s, t in zip(intent_pdf.intent_score, intent_pdf.transactions)]
ax.bar(intent_pdf.label, intent_pdf.intent_score, color=bar_colors)
ax.axhline(INTENT_THRESHOLD, linestyle="--", color="#333333", label="threshold = 5")
ax.set(title="Intent theo user-window (đỏ = đủ điều kiện trigger)",
       xlabel="User / window start", ylabel="Intent score")
ax.legend(); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
# 3.6 File publisher và Structured Streaming plan
def payload(row):
    return json.dumps({
        "timestamp": int(row.timestamp),
        "userid": int(row.userid),
        "event": row.event,
        "itemid": int(row.itemid),
        "transactionid": None if pd.isna(row.transactionid) else int(row.transactionid),
    })

def publish_file(index, rows):
    frame = pd.DataFrame(rows, columns=event_columns)
    final_path = INPUT_DIR / f"slice_{index:03d}.json"
    temp_path = INPUT_DIR / f".slice_{index:03d}.tmp"
    temp_path.write_text("\n".join(payload(r) for r in frame.itertuples(index=False)) + "\n",
                         encoding="utf-8")
    temp_path.replace(final_path)  # atomic publish
    return len(frame)

FORCE_WINDOWS_STREAMING = os.environ.get("FORCE_WINDOWS_STREAMING", "false").lower() == "true"
STREAMING_READY = os.name != "nt" or FORCE_WINDOWS_STREAMING

if STREAMING_READY:
    raw = (spark.readStream.format("text")
           .option("maxFilesPerTrigger", 1)
           .load(str(INPUT_DIR)))

    parsed = (raw.select(F.from_json("value", EVENT_SCHEMA).alias("e")).select("e.*")
        .filter(F.col("userid").isNotNull() & F.col("event").isin(*WEIGHTS))
        .withColumn("event_time", F.timestamp_millis("timestamp"))
        .withColumn("weight", weight_expr)
        .withColumn("is_view", (F.col("event") == "view").cast("int"))
        .withColumn("is_cart", (F.col("event") == "addtocart").cast("int"))
        .withColumn("is_txn", (F.col("event") == "transaction").cast("int")))

    windowed_intent = (parsed.withWatermark("event_time", WATERMARK)
        .groupBy(F.window("event_time", WINDOW, SLIDE).alias("window"), "userid", "itemid")
        .agg(F.sum("weight").alias("intent_score"),
             F.sum("is_view").alias("views"),
             F.sum("is_cart").alias("carts"),
             F.sum("is_txn").alias("transactions")))
    assert windowed_intent.isStreaming
    windowed_intent.printSchema()
else:
    print("SKIP native file streaming trên Windows (NativeIO).")
    print("Chạy trong Spark container để xem watermark, late data và checkpoint restart;")
    print("phần static + foreachBatch logic vẫn được thực thi bên dưới.")

### 3.7.1 Chọn eligible seed và khởi tạo observability

Một seed hợp lệ phải có intent đạt threshold và chưa transaction. `batch_results`/`batch_meta` chỉ là bộ nhớ quan sát cho notebook; production nên ghi vào sink idempotent.

In [ ]:
logger = logging.getLogger("a2_v3_demo")
logger.handlers.clear(); logger.setLevel(logging.INFO); logger.propagate = False
file_handler = logging.FileHandler(LOG_DIR / "micro_batches.log")
file_handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(file_handler)
batch_results, batch_meta = {}, []

def select_eligible_seeds(batch):
    return (batch.filter((F.col("intent_score") >= INTENT_THRESHOLD) & (F.col("transactions") == 0))
        .select("userid", F.col("itemid").alias("seed_item"), "window", "intent_score", "views", "carts"))

### 3.7.2 Sinh candidate và xếp hạng

Ưu tiên candidate từ item-to-item CF. Nếu seed không có neighbor, fallback sang popularity. Ranking luôn partition theo `userid` và `window`, vì cùng user ở hai window là hai context khác nhau.

In [ ]:
def rank_candidates(seeds):
    candidates = (seeds.join(F.broadcast(item_sim), seeds.seed_item == item_sim.item_a)
        .select("userid", "window", "seed_item", F.col("item_b").alias("rec_item"),
                (F.col("intent_score") * F.col("similarity")).alias("rec_score"), "intent_score", "carts"))
    rank_window = Window.partitionBy("userid", "window").orderBy(F.desc("rec_score"), F.asc("rec_item"))
    cf = (candidates.withColumn("rank", F.row_number().over(rank_window)).filter(F.col("rank") <= TOP_K)
          .withColumn("source", F.lit("item_cf")))
    missing = (seeds.select("userid", "window", "seed_item", "intent_score", "carts")
        .join(cf.select("userid", "window").distinct(), ["userid", "window"], "left_anti"))
    fallback = (missing.crossJoin(F.broadcast(popular)).filter(F.col("rec_item") != F.col("seed_item"))
        .withColumn("rank", F.row_number().over(Window.partitionBy("userid", "window").orderBy(F.desc("pop_score"), F.asc("rec_item"))))
        .filter(F.col("rank") <= TOP_K)
        .select("userid", "window", "seed_item", "rec_item", F.lit(0.0).alias("rec_score"), "intent_score", "carts", "rank")
        .withColumn("source", F.lit("popular")))
    return cf.select(*fallback.columns).unionByName(fallback)

### 3.7.3 `foreachBatch` orchestration

Hàm callback chỉ phối hợp các bước: lấy seed → tạo candidate → xếp hạng → tạo metadata/output. Khi Spark retry một batch, sink thật cần khóa idempotent để không tạo duplicate side effect.

In [ ]:
def recommend_batch(batch_df, batch_id):
    if batch_df.isEmpty(): return
    started = time.perf_counter(); batch = batch_df.persist(); changed_rows = batch.count()
    seeds = select_eligible_seeds(batch)
    recommendations = (rank_candidates(seeds).withColumn("window_start", F.col("window.start"))
        .withColumn("window_end", F.col("window.end"))
        .withColumn("trigger_reason", F.when(F.col("carts") > 0, "cart_no_purchase").otherwise("heavy_browsing"))
        .withColumn("batch_id", F.lit(int(batch_id))).drop("window", "carts").persist())
    recommendation_count = recommendations.count()
    if recommendation_count and STREAMING_READY:
        recommendations.write.mode("overwrite").partitionBy("batch_id").parquet(str(OUTPUT_DIR))
    result_pdf = recommendations.orderBy("userid", "window_end", "rank").toPandas()
    batch_results[int(batch_id)] = result_pdf
    meta = {"batch_id": int(batch_id), "changed_window_rows": changed_rows,
            "triggered_users": seeds.select("userid").distinct().count(), "recommendations": recommendation_count,
            "foreachBatch_ms": round((time.perf_counter() - started) * 1000)}
    batch_meta.append(meta); logger.info(json.dumps(meta)); print(meta)
    if not result_pdf.empty: display(result_pdf.head(12))
    recommendations.unpersist(); batch.unpersist()

## 3.8 Replay theo từng tình huống

Thay vì publish một khối event lớn, các cell dưới đây chạy **từng case**. Mỗi case có hành vi đầu vào, lý do trigger/suppress, và cell code riêng. Trên Windows native, file stream bị bỏ qua do `NativeIO`; fallback cuối phần vẫn kiểm tra bốn quy tắc business cơ bản.

In [ ]:
late_input_rows, late_dropped = None, None
if STREAMING_READY:
    def start_query(name="a2_v3_intent"):
        return (windowed_intent.writeStream.queryName(name).outputMode("update")
            .foreachBatch(recommend_batch).option("checkpointLocation", str(CHECKPOINT_DIR))
            .trigger(processingTime="1 second").start())
    def run_case(file_index, label, rows):
        print(f"\n--- {label}: {len(rows)} event(s) ---")
        publish_file(file_index, rows); query.processAllAvailable()
    query = start_query()
else:
    print("Streaming-only cases: SKIPPED trên native Windows.")

### 3.8.1 View nhiều lần → high intent

User `9002` xem item `201` năm lần trong cùng window. Score `5 × 1 = 5` đạt threshold; phải phát recommendation với `heavy_browsing`.

In [ ]:
if STREAMING_READY:
    run_case(0, "CASE 1 — five views => high intent", [
        event_at(0,9002,"view",201), event_at(.5,9002,"view",201), event_at(1,9002,"view",201),
        event_at(1.5,9002,"view",201), event_at(2,9002,"view",201)])

### 3.8.2 `addtocart` → high intent

User `9001` thêm item `101` vào giỏ. `addtocart` có trọng số 5, nên recommendation được kích hoạt ngay trong session.

In [ ]:
if STREAMING_READY:
    run_case(1, "CASE 2 — addtocart => high intent", [
        event_at(0,9001,"view",101), event_at(1,9001,"view",101), event_at(2,9001,"addtocart",101)])

### 3.8.3 `addtocart` + `transaction` → không recommendation

User `9003` có cart rồi transaction cùng item/window. `transaction` biểu thị conversion, nên rule `transactions == 0` suppress recommendation.

In [ ]:
if STREAMING_READY:
    run_case(2, "CASE 3 — converted => suppress", [
        event_at(1,9003,"addtocart",301), event_at(2,9003,"transaction",301,7001)])

### 3.8.4 Intent thấp → không trigger

User `9004` chỉ có hai view, score `2 < 5`. Threshold giúp tránh over-trigger với tín hiệu yếu.

In [ ]:
if STREAMING_READY:
    run_case(3, "CASE 4 — low intent => no trigger", [event_at(1,9004,"view",102), event_at(2,9004,"view",102)])

### 3.8.5 Đẩy watermark

Event phút 20 của user `9999` chỉ để đẩy `max(event_time)` tới 20. Với watermark 5 phút, ngưỡng late xấp xỉ phút 15.

In [ ]:
if STREAMING_READY:
    run_case(4, "Advance watermark", [event_at(20,9999,"view",101)])
    print("Current watermark:", (query.lastProgress.get("eventTime") or {}).get("watermark"))

### 3.8.6 Late event còn trong watermark

Event `addtocart` của user `9010` có event time phút 18. Vì `18 > 15`, dù đến trễ nó vẫn còn trong watermark và phải trigger.

In [ ]:
if STREAMING_READY:
    before_late_batch = query.lastProgress["batchId"]
    run_case(5, "CASE 5 — accepted late event", [event_at(18,9010,"addtocart",103)])
    late_progress = [p for p in query.recentProgress if p["batchId"] > before_late_batch]
    print("Accepted-late input rows:", sum(p["numInputRows"] for p in late_progress))

### 3.8.7 Too-late event bị watermark loại

Event của user `9011` ở phút 0, thấp hơn watermark khoảng phút 15. Spark loại event này để giới hạn state; xem metric `numRowsDroppedByWatermark`.

In [ ]:
if STREAMING_READY:
    before_too_late_batch = query.lastProgress["batchId"]
    run_case(6, "CASE 6 — too-late event", [event_at(0,9011,"addtocart",203)])
    late_progress = [p for p in query.recentProgress if p["batchId"] > before_too_late_batch]
    late_input_rows = sum(p["numInputRows"] for p in late_progress)
    late_dropped = sum(s.get("numRowsDroppedByWatermark",0) for p in late_progress for s in p.get("stateOperators",[]))
    print("Too-late input rows:", late_input_rows, "| dropped:", late_dropped)
else:
    recommend_batch(static_intent, 0)
    print("Static fallback complete; true watermark requires Docker/Linux.")

## 3.9 Observability theo micro-batch

Tách source metrics (`input_rows`), state metrics (`state_rows`, `late_dropped`) và business metrics (`triggered_users`). Một event có thể cập nhật nhiều sliding window, nên không diễn giải số event như số recommendation.

In [ ]:
# 3.9 Observability và visualization theo micro-batch
meta_df = pd.DataFrame(batch_meta)

if STREAMING_READY:
    progress_rows = [{
        "batch_id": p["batchId"],
        "input_rows": p["numInputRows"],
        "trigger_ms": p.get("durationMs", {}).get("triggerExecution", 0),
        "watermark": (p.get("eventTime") or {}).get("watermark"),
        "state_rows": sum(s.get("numRowsTotal", 0)
                          for s in p.get("stateOperators", [])),
        "late_dropped": sum(s.get("numRowsDroppedByWatermark", 0)
                            for s in p.get("stateOperators", [])),
    } for p in query.recentProgress]
    progress_df = pd.DataFrame(progress_rows)
    batch_report = progress_df.merge(meta_df, on="batch_id", how="left")
    display(batch_report)

    fig, axes = plt.subplots(1, 3, figsize=(13, 3))
    axes[0].bar(progress_df.batch_id, progress_df.input_rows, color="#2878b5")
    axes[0].set(title="Input rows", xlabel="Batch ID")
    axes[1].plot(progress_df.batch_id, progress_df.state_rows,
                 marker="o", color="#59a14f")
    axes[1].set(title="State rows", xlabel="Batch ID")
    axes[2].bar(meta_df.batch_id, meta_df.triggered_users, color="#f28e2b")
    axes[2].set(title="Triggered users", xlabel="Batch ID")
    for axis in axes:
        axis.grid(axis="y", alpha=.2)
    plt.tight_layout(); plt.show()
else:
    display(meta_df)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(meta_df.batch_id, meta_df.triggered_users, color="#f28e2b")
    ax.set(title="Triggered users — static micro-batch fallback",
           xlabel="Batch ID", ylabel="Users")
    ax.grid(axis="y", alpha=.2); plt.tight_layout(); plt.show()

## 3.10 Checkpoint restart

Query dừng rồi khởi động lại với cùng `checkpointLocation`. Không publish data mới ngay sau restart: data đã commit không được đọc lại. Sau đó gửi user `9020`; chỉ event mới được xử lý. Đây là checkpoint recovery; external sink vẫn cần idempotency để đạt effectively-once.

In [ ]:
rows_reread_after_restart, new_data_rows = None, None
if STREAMING_READY:
    last_batch_before_stop = query.lastProgress["batchId"]; query.stop()
    query = start_query("a2_v3_intent_restarted"); query.processAllAvailable()
    rows_reread_after_restart = sum(p["numInputRows"] for p in query.recentProgress)
    print("Rows đọc lại ngay sau restart:", rows_reread_after_restart)
    run_case(7, "CASE 7 — new event after restart", [event_at(22,9020,"addtocart",202)])
    new_data_rows = sum(p["numInputRows"] for p in query.recentProgress)
    print("Rows sau event mới:", new_data_rows, "| batch trước stop:", last_batch_before_stop)
    query.stop()
else:
    print("Checkpoint restart demo cần Docker/Linux file streaming.")

## 3.11 Validation: một assertion cho mỗi khái niệm

Nếu assertion fail, quay lại đúng case ở trên: case 1/2 trigger, case 3/4 suppress, case 5/6 watermark, case 7 checkpoint.

In [ ]:
# 3.11 Validation: mỗi assertion gắn với một khái niệm
non_empty = [frame for frame in batch_results.values() if not frame.empty]
if not non_empty:
    raise AssertionError("Demo chưa tạo recommendation nào")
all_recommendations = pd.concat(non_empty, ignore_index=True)

users_recommended = set(all_recommendations.userid.astype(int))
assert 9001 in users_recommended, "addtocart/high intent phải trigger"
assert 9002 in users_recommended, "5 views phải trigger heavy browsing"
assert 9003 not in users_recommended, "transaction trong window phải suppress"
assert 9004 not in users_recommended, "intent dưới threshold không được trigger"

rank_counts = (all_recommendations
    .groupby(["userid", "window_end", "batch_id"]).size())
assert (rank_counts <= TOP_K).all(), "Mỗi user-window chỉ được top-K"

print("PASS — addtocart/high-intent trigger")
print("PASS — heavy browsing trigger")
print("PASS — transaction suppression")
print("PASS — below-threshold suppression")
print("PASS — top-K")

if STREAMING_READY:
    assert 9010 in users_recommended, "late event còn trong watermark phải được nhận"
    assert 9011 not in users_recommended, "too-late event không được tạo recommendation"
    assert rows_reread_after_restart == 0, "restart không được đọc lại file cũ"
    assert new_data_rows >= 1, "event mới sau restart phải được xử lý"
    assert late_dropped > 0, "late demo phải ghi nhận row/window bị watermark loại"
    print("PASS — accepted-late và too-late behaviour")
    print("PASS — checkpoint restart")
else:
    print("SKIP — late-data/checkpoint assertions cần Docker/Linux file streaming")

display(all_recommendations.sort_values(
    ["batch_id", "userid", "window_end", "rank"]).head(30))

<a id="appendix"></a>
# Phụ lục — Liên hệ với Part B

| A2 teaching demo | Part B implementation |
|---|---|
| Synthetic JSON files | Retail Rocket `events.csv` |
| File replay | Kafka producer + topic có ≥2 partitions |
| Generic `userid` | Retail Rocket `visitorid` |
| Small Item-CF | Model/fallback huấn luyện từ historical subset |
| Local `foreachBatch` output | Console + partitioned Parquet/Delta |
| Controlled late events | Real out-of-order clickstream |
| Local checkpoint | Durable checkpoint và restart evidence |
| Simple top-K | Seen-item exclusion, event-time cooldown, idempotent sent-log |

Part B chịu trách nhiệm chứng minh Kafka ingestion, topic partitions, real/subset data, persistent sink và live classroom demo. Không nên dùng kết quả synthetic của A2 như kết luận thực nghiệm về Retail Rocket.

<a id="sec-4"></a>
# 4. Kết luận

- Recommender system gồm candidate generation, scoring, ranking, serving và feedback.
- Real-time recommender thường kết hợp offline model với live context/online scoring.
- Streaming cải thiện freshness và phản ứng trong session; nó không tự động thay thế batch training.
- Structured Streaming dùng event time, sliding window, watermark và managed state để tính live intent.
- High-intent item trở thành seed; item similarity và top-K ranking tạo recommendation trước khi session kết thúc.
- Demo đã minh họa transaction suppression, late data, state metrics và checkpoint restart bằng synthetic data có kiểm soát.

<a id="sec-5"></a>
# 5. Tài liệu tham khảo

1. Armbrust, M. et al. (2018). *Structured Streaming: A Declarative API for Real-Time Applications*. SIGMOD.
2. Zaharia, M. et al. (2013). *Discretized Streams: Fault-Tolerant Streaming Computation at Scale*. SOSP.
3. Linden, G., Smith, B., & York, J. (2003). *Amazon.com Recommendations: Item-to-Item Collaborative Filtering*. IEEE Internet Computing.
4. Hu, Y., Koren, Y., & Volinsky, C. (2008). *Collaborative Filtering for Implicit Feedback Datasets*. ICDM.
5. Koren, Y., Bell, R., & Volinsky, C. (2009). *Matrix Factorization Techniques for Recommender Systems*. Computer.
6. Hidasi, B. et al. (2016). *Session-based Recommendations with Recurrent Neural Networks*. ICLR.
7. Ludewig, M., & Jannach, D. (2018). *Evaluation of Session-based Recommendation Algorithms*. UMUAI.
8. Covington, P., Adams, J., & Sargin, E. (2016). *Deep Neural Networks for YouTube Recommendations*. RecSys.
9. Apache Spark. *Structured Streaming Programming Guide* và *Kafka Integration Guide*.

In [ ]:
# Giải phóng tài nguyên
item_sim.unpersist()
popular.unpersist()
spark.stop()
print("Spark session đã dừng.")